# Amazon Product Query Assistant
End-to-end RAG system over Amazon All_Beauty products (112K products, 701K reviews)

In [19]:
import os
os.getcwd()
os.chdir('/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/')

In [1]:
import sys
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
sys.path.append(".")

## 2. Data Loading

In [2]:
# Load raw data from HuggingFace (first time only)
from pathlib import Path

if Path("data/raw/reviews_raw.jsonl").exists():
    print("Loading from local files...")
    reviews = [json.loads(l) for l in open("data/raw/reviews_raw.jsonl")]
    meta    = [json.loads(l) for l in open("data/raw/meta_raw.jsonl")]
    print(f"Reviews: {len(reviews):,}, Meta: {len(meta):,}")
else:
    print("Downloading from HuggingFace...")
    from datasets import load_dataset
    reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", split="full", trust_remote_code=True)
    meta    = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_All_Beauty", split="full", trust_remote_code=True)
    reviews = list(reviews)
    meta    = list(meta)
    # save locally
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    with open("data/raw/reviews_raw.jsonl", "w") as f:
        for row in reviews: f.write(json.dumps(dict(row)) + "\n")
    with open("data/raw/meta_raw.jsonl", "w") as f:
        for row in meta: f.write(json.dumps(dict(row)) + "\n")
    print(f"Saved. Reviews: {len(reviews):,}, Meta: {len(meta):,}")

/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-03 14:08:28,217 PyTorch version 2.2.0 available.


Saved. Reviews: 701,528, Meta: 112,590


## 3. Preprocessing

In [7]:
# Build merged product documents (run once, loads from file after)
if Path("data/processed/products.jsonl").exists():
    print("Loading processed products...")
    with open("data/processed/products.jsonl", "r") as f:
        products = [json.loads(line) for line in f if line.strip()]
    print(f"Products loaded: {len(products):,}")
else:
    from src.preprocessor import build_products
    products = build_products(meta, reviews)
    print(f"Products built: {len(products):,}")

Loading processed products...
Products loaded: 112,590


## 4. BM25 Retrieval

In [8]:
# Build corpus
from src.utils import build_corpus
corpus, tokenized_corpus = build_corpus(products)

2026-05-03 14:10:03,462 Building corpus from 112,590 products...
2026-05-03 14:10:05,971 Corpus built: 112,590 documents


In [9]:
# Load or build BM25 index
from src.bm25 import build_bm25, load_bm25, search_bm25

if Path("data/processed/bm25_index.pkl").exists():
    bm25, _ = load_bm25()
    print("BM25 index loaded!")
else:
    bm25 = build_bm25(tokenized_corpus)
    print("BM25 index built!")

2026-05-03 14:10:05,977 Loading BM25 index from data/processed/bm25_index.pkl...
2026-05-03 14:10:06,709 BM25 index loaded.


BM25 index loaded!


In [10]:
# Test BM25
results = search_bm25(bm25, products, "moisturizer for sensitive skin", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title'][:70]}")

16.8432 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pa
16.0985 | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With B
15.4566 | Revitalizing Light-Weight Moisturizer SPF 15
14.9437 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
14.5508 | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itchy Skin


## 5. Semantic Retrieval

In [11]:
# Load or build semantic index
from src.semantic import build_semantic_index, load_semantic_index, search_semantic
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

if Path("data/processed/faiss.index").exists():
    index, embeddings = load_semantic_index()
    print("Semantic index loaded!")
else:
    index, embeddings = build_semantic_index(corpus)
    print("Semantic index built!")

2026-05-03 14:10:16,139 Loading faiss.
2026-05-03 14:10:16,192 Successfully loaded faiss.
2026-05-03 14:10:17,240 Use pytorch device_name: mps
2026-05-03 14:10:17,240 Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-05-03 14:10:18,803 Loading FAISS index from data/processed/faiss.index...
2026-05-03 14:10:18,920 FAISS index loaded: 112590 vectors


Semantic index loaded!


In [12]:
# Test semantic search
results = search_semantic(index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"{r['semantic_score']:.4f} | {r['title'][:70]}")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.07s/it]

0.7514 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
0.7011 | Living Nature Sensitive Skin Day Moisture Cream
0.6976 | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For Daily Use 
0.6934 | Moisturel Therapeutic Lotion 14 oz (Pack of 3)
0.6926 | Cetaphil Moisturizing Cream 20 oz, 2 Pack


## 6. Hybrid Retrieval

In [13]:
from src.hybrid import hybrid_search

results = hybrid_search(bm25, index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"hybrid={r['hybrid_score']:.6f} | bm25={r['bm25_rank']} | sem={r['semantic_rank']} | {r['title'][:60]}")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 13.52it/s]

hybrid=0.031818 | bm25=0 | sem=6 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 
hybrid=0.030952 | bm25=None | sem=0 | Senka Perfect emulsion Silky Moisture moisturizing lotion 15
hybrid=0.030679 | bm25=None | sem=1 | Living Nature Sensitive Skin Day Moisture Cream
hybrid=0.030679 | bm25=1 | sem=None | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive S
hybrid=0.030415 | bm25=None | sem=2 | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For 


## 7. Evaluation

In [14]:
# Run all 10 evaluation queries
queries = [
    "moisturizer for sensitive skin",
    "vitamin c serum",
    "shampoo for curly hair",
    "something to keep my skin hydrated all day",
    "product for damaged hair",
    "gentle face wash for acne",
    "best affordable moisturizer under $20 for dry skin",
    "natural organic hair care for color treated hair",
    "anti aging cream for women over 50",
    "fragrance free products for baby sensitive skin",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    print("\nBM25:")
    for r in search_bm25(bm25, products, query, top_k=5):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")

    print("\nSemantic:")
    for r in search_semantic(index, products, query, top_k=5, model=model):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")

    print("\nHybrid:")
    for r in hybrid_search(bm25, index, products, query, top_k=5, model=model):
        print(f"  {r['parent_asin']} | {r['title'][:60]}")


QUERY: moisturizer for sensitive skin

BM25:
  B00YF3J4OG | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 
  B072BKHJ7Z | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive S
  B07F2P45NZ | Revitalizing Light-Weight Moisturizer SPF 15
  B0842CC6X9 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
  B07B8PHZPM | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itch

Semantic:


Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 30.18it/s]


  B01JFPP3L6 | Senka Perfect emulsion Silky Moisture moisturizing lotion 15
  B014T3QBLK | Living Nature Sensitive Skin Day Moisture Cream
  B0757RW9XK | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For 
  B01IADYIM4 | Moisturel Therapeutic Lotion 14 oz (Pack of 3)
  B09PNWWBYX | Cetaphil Moisturizing Cream 20 oz, 2 Pack

Hybrid:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 154.80it/s]

  B00YF3J4OG | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 
  B01JFPP3L6 | Senka Perfect emulsion Silky Moisture moisturizing lotion 15
  B014T3QBLK | Living Nature Sensitive Skin Day Moisture Cream
  B072BKHJ7Z | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive S
  B0757RW9XK | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For 

QUERY: vitamin c serum

BM25:


...

  B010L93H2C | Best Anti Aging Cream For Men - Protect And Nourish Damaged 
  B0109SSSYC | Spa Ultimate Retinol Anti Aging Night Cream 1.69 Fluid Ounce
  B01CS3OWBY | Allegro Anti Aging Cream - Anti-Wrinkle Cream - Anti Aging P
  B073WCK5H7 | Best Selling Anti-Aging Skin Care Kits: Krasa Anti-Aging Cre
  B01JZTITS6 | Derm Essence Anti-Aging Cream 1.0 Fl Oz/30mL

QUERY: fragrance free products for baby sensitive skin

BM25:
  B07NXW5397 | Huggies Natural Care Baby Wipe Refill, Fragrance Free (1,040
  B0B7MCJ618 | Parents Choice Fragrance Free Baby Wipes, 1200 Count (2 Pack
  B08HSH5VSW | Little Journey Sensitive Baby Wipes - Value Pack 192ct
  B07L52XF7G | Simple Truth Fragrance Free Baby Wipes 4 Pack 256 ct
  B09HF6TGBR | Love Labs Organics 4oz. Baby Body Butter. All Natural Safe a

Semantic:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 146.39it/s]


  B07L52XF7G | Simple Truth Fragrance Free Baby Wipes 4 Pack 256 ct
  B088RL8P6G | Babyology All Natural Baby Wash and Shampoo - 100% Edible In
  B001LBH1T0 | Sweet & Soft Baby Fragrance - Kids Fragrance - Perfect Size 
  B088RKZMLX | Babyology All Natural Baby Wash and Shampoo - 100% Edible In
  B07L443CKY | Babyology - All Natural Baby Wash and Shampoo - 100% Edible 

Hybrid:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 123.89it/s]

  B07L52XF7G | Simple Truth Fragrance Free Baby Wipes 4 Pack 256 ct
  B07NXW5397 | Huggies Natural Care Baby Wipe Refill, Fragrance Free (1,040
  B0B7MCJ618 | Parents Choice Fragrance Free Baby Wipes, 1200 Count (2 Pack
  B088RL8P6G | Babyology All Natural Baby Wash and Shampoo - 100% Edible In
  B088RKZMLX | Babyology All Natural Baby Wash and Shampoo - 100% Edible In


In [15]:
ground_truth = {
    "moisturizer for sensitive skin": [
        "B00YF3J4OG", "B072BKHJ7Z", "B07B8PHZPM",
        "B014T3QBLK", "B09PNWWBYX", "B01IADYIM4", "B0757RW9XK"
    ],
    "vitamin c serum": [
        "B00HSX95A8", "B07NPX9LNS", "B071177MV1",
        "B09987JTXF", "B00VQHFBBE", "B01CM8G8PI",
        "B016MNB4HQ", "B019D6WNJM", "B00S5LB08W", "B000MXRFY4"
    ],
    "shampoo for curly hair": [
        "B07L46W7J4",  # Maui Moisture Curl Quench Shampoo
        "B0884Z6NHP",  # DermOrganic Curl Shampoo Sulfate Free
        "B01M09LKNF",  # Back To Basics Curl Enhancing Shampoo
        "B01B6Z1J52",  # Curly Hair Shampoo and Conditioner Set
        "B00BUQ475W",  # Marc Anthony Strictly Curls Shampoo
        "B07TDB2M5R",  # LaCoupe Naturals for Curly Hair
    ],
    "something to keep my skin hydrated all day": [
        "B00ZM9A5NG",  # SkinResource Moisture Boost Hydragel
        "B07YLSHVXC",  # uruoi Deep Moisture Gel
        "B008JBVDUS",  # MD Formulations Moisture Defense Cream
        "B00T45V7JU",  # Curel Daily Healing Lotion
        "B00LZGWJJS",  # Ultimate Healing Moisturizing Lotion
        "B06W9L29W9",  # Hyaluronic Moisturizer Cream
    ],
    "product for damaged hair": [
        "B076J3YR85",  # AQUADERMA Hair Repairing Treatment for Damaged Hair
        "B00FN3GF6M",  # Active Organic Sea Buckthorn Oil for Damaged Hair
        "B0082Q097S",  # Avon Damage Repair 5-Day Rescue Treatment
        "B07VN6GWXP",  # Toni & Guy Damage Repair Shampoo and Conditioner
        "B013IRN9E0",  # Hair Repair Mask with Argan Oil
        "B00LCBLEGK",  # Oribe Gold Lust Transformative Masque
    ],
    "gentle face wash for acne": [
        "B001E96OGK",  # Neutrogena Oil-Free Acne Wash
        "B07T4JF9M7",  # Gentle Facial Cleanser with Rose Water
        "B019769J2W",  # Detox Handcrafted Face Wash
        "B014EUKL6O",  # Organic Face Wash By BeeFriendly
        "B0092T3OB2",  # OXY Clinical Advanced Face Wash Acne Treatment
        "B01BP5L8TA",  # 808 Dude Face Wash for Teen Acne
    ],
    "best affordable moisturizer under $20 for dry skin": [
        "B018TZTS96",  # Face Moisturizer for Dry Skin by WONDERPIEL
        "B01M1A0H4Z",  # DayTime Moisturizer for Dry Skin
        "B01IADYIM4",  # Moisturel Therapeutic Lotion
        "B00H221BSO",  # Lubrisoft Moisture Lotion for Dry Skin
        "B0757RW9XK",  # Daily Face Moisturizer for Dry Skin
    ],
    "natural organic hair care for color treated hair": [
        "B07BR5QSTK",  # Organix Sulfate Free Color Reviving
        "B01GOWKAD4",  # Argan Oil Shampoo Sulfate Free Natural
        "B0875YT39N",  # Woman to Woman Naturals Organic Hair Shampoo
        "B07TBB52VW",  # LaCoupe Naturals for Damaged Hair Sulfate Free
        "B07NH66JHX",  # Color Boost Brown Color Depositing Shampoo
    ],
    "anti aging cream for women over 50": [
        "B0109SSSYC",  # Spa Ultimate Retinol Anti Aging Night Cream
        "B01CS3OWBY",  # Allegro Anti Aging Cream
        "B01N0WY90C",  # Anti Aging Night Cream Hydrating Face Cream
        "B073WCK5H7",  # Krasa Anti-Aging Cream Kit
        "B01JZTITS6",  # Derm Essence Anti-Aging Cream
        "B00FJPMNEM",  # Anti Aging Vitamin C Cream
    ],
    "fragrance free products for baby sensitive skin": [
        "B07NXW5397",  # Huggies Natural Care Baby Wipe Fragrance Free
        "B0B7MCJ618",  # Parents Choice Fragrance Free Baby Wipes
        "B07L52XF7G",  # Simple Truth Fragrance Free Baby Wipes
        "B088RL8P6G",  # Babyology All Natural Baby Wash
        "B088RKZMLX",  # Babyology All Natural Baby Wash
        "B09HF6TGBR",  # Love Labs Organics Baby Body Butter
    ],
}

In [21]:
import importlib
import sys

if 'src.retrieval_metrics' in sys.modules:
    del sys.modules['src.retrieval_metrics']

from src.retrieval_metrics import evaluate_all

results = evaluate_all(bm25, index, products, ground_truth, model, k=5)

for method in ["bm25", "semantic", "hybrid"]:
    print(f"\n{method.upper()} Average:")
    for metric, score in results[method]["average"].items():
        print(f"  {metric}: {score}")

Batches: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 176.78it/s]
2026-05-03 14:14:45,569 BM25 average: {'precision@5': 0.6, 'mrr': 0.75, 'ndcg@5': 0.6002}
2026-05-03 14:14:45,570 SEMANTIC average: {'precision@5': 0.66, 'mrr': 0.825, 'ndcg@5': 0.6912}
2026-05-03 14:14:45,570 HYBRID average: {'precision@5': 0.78, 'mrr': 0.725, 'ndcg@5': 0.7147}



BM25 Average:
  precision@5: 0.6
  mrr: 0.75
  ndcg@5: 0.6002

SEMANTIC Average:
  precision@5: 0.66
  mrr: 0.825
  ndcg@5: 0.6912

HYBRID Average:
  precision@5: 0.78
  mrr: 0.725
  ndcg@5: 0.7147


## 8. RAG Pipeline

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("OPENAI_API_KEY")[:10])

sk-proj-YC


In [24]:
if 'src.tools' in sys.modules:
    del sys.modules['src.tools']

from src.tools import detect_filters, apply_tools

# test filter detection
print(detect_filters("best moisturizer under $20"))
print(detect_filters("serum rated above 4 stars"))
print(detect_filters("best affordable cream under $30 rating 4.5"))

# test apply tools on retrieved products
retrieved = hybrid_search(bm25, index, products, "moisturizer under $20", top_k=10, model=model)
filtered, filters = apply_tools(retrieved, "moisturizer under $20")
print(f"\nFilters detected: {filters}")
print(f"Before: {len(retrieved)} → After: {len(filtered)} products")
for p in filtered:
    print(f"  ${p['price']} | {p['title'][:60]}")

2026-05-03 19:50:27,930 Detected filters: {'max_price': 20.0}
2026-05-03 19:50:27,931 Detected filters: {'min_rating': 4.0}
2026-05-03 19:50:27,931 Detected filters: {'max_price': 30.0, 'min_rating': 4.5}


{'max_price': 20.0}
{'min_rating': 4.0}
{'max_price': 30.0, 'min_rating': 4.5}


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.33it/s]
2026-05-03 19:50:30,872 Detected filters: {'max_price': 20.0}
2026-05-03 19:50:30,873 filter_by_price(max=20.00): 10 → 1 products



Filters detected: {'max_price': 20.0}
Before: 10 → After: 1 products
  $20.0 | Beautederm Night Cream #2 (Moisturizer), 20 g. by Koo Koo 4 


In [26]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']

from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)
print(result["response"])

ModuleNotFoundError: No module named 'openai'

In [29]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']

from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)
print(result["response"])

2026-05-04 14:46:30,862 RAG query: best moisturizer for sensitive skin
Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.37it/s]
2026-05-04 14:46:33,692 Retrieved 5 products
2026-05-04 14:46:40,627 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-04 14:46:40,659 Answer generated.


For sensitive skin, I recommend the following moisturizers from the list:

1. **Simple Hydrating Light Moisturizer, for Sensitive Skin** - 4.2 Ounce (Pack of 3)  
   - **Price:** Not specified  
   - **Rating:** 5.0 (2 reviews)  
   - **Top Review:** "This lotion is a life saver and it's amazing. You will see results instantly."

2. **Senka Perfect Emulsion Silky Moisture Moisturizing Lotion** - 150ml  
   - **Price:** Not specified  
   - **Rating:** 4.0 (67 reviews)  
   - **Top Review:** "I have sensitive skin and this really helps my skin. I didn’t have reactions and my skin is glowing."

3. **Simple Hydrating Light Moisturizer** - 4.2 Ounce (6-pack)  
   - **Price:** Not specified  
   - **Rating:** 5.0 (12 reviews)  
   - **Top Review:** "This moisturizer is very hydrating but very light and non-greasy. It is the best for sensitive skin."

These options have high ratings and positive reviews specifically mentioning their effectiveness for sensitive skin.


## 9. Guardrails

In [32]:
if 'src.guardrails' in sys.modules:
    del sys.modules['src.guardrails']

from src.guardrails import check_input, check_output

queries = [
    "best moisturizer for sensitive skin",
    "how to make a bomb",
    "what is the weather today",
    "asdfjkl;",
    "hi",
    "best laptop under $500",
]

for q in queries:
    result = check_input(q)
    status = "✅" if result["valid"] else "❌"
    print(f"{status} '{q[:40]}' → {result['reason'] or 'valid'}")

2026-05-04 14:51:07,369 Input guardrail passed: best moisturizer for sensitive skin


✅ 'best moisturizer for sensitive skin' → valid
❌ 'how to make a bomb' → Query contains inappropriate content.
❌ 'what is the weather today' → This assistant only answers questions about beauty and personal care products. Please ask about skincare, haircare, or similar topics.
❌ 'asdfjkl;' → Query is too short. Please be more specific.
❌ 'hi' → Query is too short. Please be more specific.
❌ 'best laptop under $500' → This assistant only answers questions about beauty and personal care products. Please ask about skincare, haircare, or similar topics.


In [34]:
from src.rag import rag_answer

result = rag_answer(
    query="best moisturizer for sensitive skin",
    bm25=bm25,
    index=index,
    products=products,
    model=model,
)

# now test output guardrails
from src.guardrails import check_output

print(check_output(result["response"], result["retrieved"]))
print(check_output("This product cures eczema and treats acne permanently.", result["retrieved"]))
print(check_output("I don't know.", result["retrieved"]))

2026-05-04 14:52:10,840 RAG query: best moisturizer for sensitive skin
Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.87it/s]
2026-05-04 14:52:11,452 Retrieved 5 products
2026-05-04 14:52:16,277 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-04 14:52:16,301 Answer generated.
2026-05-04 14:52:16,306 Output guardrail passed.


{'valid': True, 'reason': None}
{'valid': False, 'reason': 'Answer is too short or uninformative.'}
{'valid': False, 'reason': 'Answer is too short or uninformative.'}


In [35]:
from src.guardrails import MEDICAL_CLAIM_KEYWORDS
print(MEDICAL_CLAIM_KEYWORDS)

['cures', 'cure', 'treats', 'treatment for', 'heals', 'healing', 'clinically proven', 'fda approved', 'eliminates', 'prevents cancer', 'prevents disease', 'medical treatment']


In [36]:
print(check_output(
    "This product cures eczema and treats acne permanently. It has been clinically proven to heal all skin conditions and eliminate wrinkles completely within days of use.",
    result["retrieved"]
))

{'valid': False, 'reason': 'Answer contains medical claims which cannot be verified.'}


In [37]:
if 'src.rag' in sys.modules:
    del sys.modules['src.rag']
from src.rag import rag_answer

# should be blocked
result = rag_answer("best laptop under $500", bm25, index, products, model)
print("Blocked:", result["response"])

# should work
result = rag_answer("best moisturizer for sensitive skin", bm25, index, products, model)
print("Valid:", result["response"][:100])

2026-05-04 14:57:37,279 RAG query: best laptop under $500
2026-05-04 14:57:37,280 Input guardrail blocked: This assistant only answers questions about beauty and personal care products. Please ask about skincare, haircare, or similar topics.
2026-05-04 14:57:37,281 RAG query: best moisturizer for sensitive skin
2026-05-04 14:57:37,281 Input guardrail passed: best moisturizer for sensitive skin


Blocked: This assistant only answers questions about beauty and personal care products. Please ask about skincare, haircare, or similar topics.


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.34it/s]
2026-05-04 14:57:37,743 Retrieved 5 products
2026-05-04 14:57:47,814 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-04 14:57:47,819 Answer generated.
2026-05-04 14:57:47,820 Output guardrail passed.


Valid: For sensitive skin, I recommend the following moisturizers:

1. **Simple Hydrating Light Moisturizer


In [42]:
if 'src.ragas_eval' in sys.modules:
    del sys.modules['src.ragas_eval']

from src.ragas_eval import run_ragas_evaluation

eval_queries = [
    "best moisturizer for sensitive skin",
    "vitamin c serum",
    "gentle face wash for acne",
]

scores = run_ragas_evaluation(eval_queries, bm25, index, products, model)

2026-05-04 16:01:12,000 Running RAGAS evaluation on 3 queries...
2026-05-04 16:01:12,001 Evaluating query: best moisturizer for sensitive skin
2026-05-04 16:01:12,001 RAG query: best moisturizer for sensitive skin
2026-05-04 16:01:12,001 Input guardrail passed: best moisturizer for sensitive skin
Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.90it/s]
2026-05-04 16:01:12,706 Retrieved 5 products
2026-05-04 16:01:17,451 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-04 16:01:17,454 Answer generated.
2026-05-04 16:01:17,455 Output guardrail passed.
2026-05-04 16:01:17,455 Evaluating query: vitamin c serum
2026-05-04 16:01:17,455 RAG query: vitamin c serum
2026-05-04 16:01:17,456 Input guardrail passed: vitamin c serum
Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 34.36it/s]
2026-05-04 16:01:17,601 Retrieved 5 pro


── RAGAS Evaluation Results ────────────────
  Faithfulness:     0.5714
  Answer Relevancy: 0.596
────────────────────────────────────────────

